# HF Dataset Access And Browser

This notebook is the notebook companion to the Streamlit browser in
[app.py](./app.py) for the Hugging Face dataset:

- `anonymous-neurips-2026-ED/deception-localization`

It has two goals:

1. Show how to browse the dataset structure and load example files from Hugging Face.
2. Provide an interactive sentence-localization browser in notebook form, with visuals aligned to the dashboard view.

The important performance detail is that this notebook does **not** list the entire 100k-example repo by default.
Instead, it first selects an environment and model, then scans only that `environment/model/localization`
folder with a configurable cap.


## Requirements

If you do not already have these installed in your notebook kernel:

```bash
pip install huggingface_hub ipywidgets pandas numpy matplotlib
```

If widget controls do not render, also make sure your Jupyter environment has `ipywidgets` enabled.


In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd

from huggingface_hub import login, whoami
from IPython.display import HTML, Markdown, clear_output, display

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = next((candidate for candidate in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents] if (candidate / 'DatasetAccess').exists() and (candidate / 'LocalizationScripts').exists()), NOTEBOOK_ROOT)
DATASET_ACCESS_ROOT = REPO_ROOT / 'DatasetAccess'
if str(DATASET_ACCESS_ROOT) not in sys.path:
    sys.path.insert(0, str(DATASET_ACCESS_ROOT))

from hf_dataset_browser_lib import (
    DEFAULT_REPO_ID,
    DEFAULT_REPO_TYPE,
    DEFAULT_SCAN_LIMIT,
    build_generation_sentence_labels,
    build_sentence_spans,
    build_stats,
    compute_deceptive_sentence_idx,
    flatten_history,
    generation_bucket,
    list_environments,
    list_localization_repo_paths,
    list_models_for_environment,
    load_repo_record,
    normalize_history,
    parse_repo_path,
    plot_sentence_localization,
    preview_value,
    render_highlighted_sentences_html,
    render_prefix_generation_with_sentence_indices_html,
    resolve_sentence_span,
    sentence_selector_label,
    strip_code_fences,
    summarize_record_schema,
)

REPO_ID = DEFAULT_REPO_ID
REPO_TYPE = DEFAULT_REPO_TYPE
HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_CACHE_DIR = None

SELECTED_ENVIRONMENT = None
SELECTED_MODEL = None
MAX_FILES_TO_SCAN = DEFAULT_SCAN_LIMIT
EXAMPLE_FILENAME_FILTER = ""

print("Dataset repo:", REPO_ID)
print("DatasetAccess root:", DATASET_ACCESS_ROOT)


## Authenticate

If the dataset is private or attached to an anonymous review account, set `HF_TOKEN` above
or export it in your shell before running this cell.


In [ ]:
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

try:
    identity = whoami(token=HF_TOKEN or None)
    print("Authenticated as:", identity.get("name") or identity.get("fullname") or "unknown")
except Exception as exc:
    print("Authentication not confirmed yet.")
    print("If the dataset is private, set HF_TOKEN before continuing.")
    print(type(exc).__name__ + ":", exc)


## Browse The Repo Quickly

The next cell does the fast version of repo browsing:

- list environments
- list models for the chosen environment
- list only the first `MAX_FILES_TO_SCAN` localization files for the chosen environment/model


In [ ]:
environments = list_environments(repo_id=REPO_ID, repo_type=REPO_TYPE, token=HF_TOKEN or None)
print("Environments:", environments)

if not environments:
    raise RuntimeError("No environments found in the dataset repo.")

if SELECTED_ENVIRONMENT is None:
    SELECTED_ENVIRONMENT = environments[0]

models = list_models_for_environment(
    SELECTED_ENVIRONMENT,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=HF_TOKEN or None,
)
print("Models for environment:", SELECTED_ENVIRONMENT)
print(models)

if not models:
    raise RuntimeError(f"No models found for environment: {SELECTED_ENVIRONMENT}")

if SELECTED_MODEL is None:
    SELECTED_MODEL = models[0]

repo_paths = list_localization_repo_paths(
    SELECTED_ENVIRONMENT,
    SELECTED_MODEL,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=HF_TOKEN or None,
    limit=MAX_FILES_TO_SCAN,
    filename_filter=EXAMPLE_FILENAME_FILTER,
)

print(f"Selected environment: {SELECTED_ENVIRONMENT}")
print(f"Selected model: {SELECTED_MODEL}")
print(f"Listed example files: {len(repo_paths):,}")
display(pd.Series(repo_paths[:20], name="repo_path").to_frame())


## Inspect One Example

This shows the structure of one localization file and the fields inside each `history` row and `generations` row.


In [ ]:
sample_repo_path = repo_paths[0]
sample_record = load_repo_record(
    sample_repo_path,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=HF_TOKEN or None,
    cache_dir=HF_CACHE_DIR,
)

print("Sample repo path:")
print(sample_repo_path)
print()
print("Top-level keys:")
print(sorted(sample_record.keys()))

top_level_df, history_df, generation_df = summarize_record_schema(sample_record)
display(Markdown("### Top-level fields"))
display(top_level_df)
display(Markdown("### Fields inside each `history` row"))
display(history_df)
display(Markdown("### Fields inside each `generations` row"))
display(generation_df)

history = sample_record.get("history") or []
sample_probe = history[0] if history else {}
sample_generations = sample_probe.get("generations") or []

example_summary = pd.Series(
    {
        "repo_path": sample_repo_path,
        "game": sample_record.get("game"),
        "example_id": sample_record.get("example_id"),
        "prompt_len_chars": len(sample_record.get("prompt") or ""),
        "raw_text_len_chars": len(sample_record.get("raw_text") or ""),
        "history_len": len(history),
        "first_probe_sentence": sample_probe.get("sentence_text"),
        "first_probe_deception_rate": sample_probe.get("deception_rate"),
        "first_probe_num_valid": sample_probe.get("num_valid"),
        "first_probe_n_generations": len(sample_generations),
        "truthful_generations_in_first_probe": sum(g.get("is_truthful") is True for g in sample_generations),
        "deceptive_generations_in_first_probe": sum(g.get("is_truthful") is False for g in sample_generations),
        "invalid_generations_in_first_probe": sum(g.get("is_truthful") is None for g in sample_generations),
    }
)
display(example_summary.to_frame("value"))

eval_context = sample_record.get("eval_context")
if eval_context:
    display(Markdown("### Example `eval_context`"))
    display(eval_context)


## Interactive Browser

In [ ]:
browser_state = {
    "record": None,
    "history_norm": None,
    "df_plot": None,
    "df_stats": None,
    "sentence_spans": None,
    "sentence_span_map": None,
    "sample_selection_context": None,
}

env_widget = widgets.Dropdown(
    description="Environment",
    options=environments,
    value=SELECTED_ENVIRONMENT,
    layout=widgets.Layout(width="380px"),
)
model_widget = widgets.Dropdown(description="Model", layout=widgets.Layout(width="520px"))
scan_limit_widget = widgets.BoundedIntText(
    description="Scan cap",
    value=int(MAX_FILES_TO_SCAN),
    min=50,
    max=10000,
    step=50,
    layout=widgets.Layout(width="220px"),
)
example_widget = widgets.Dropdown(description="Example", layout=widgets.Layout(width="1120px"))
sentence_widget = widgets.Dropdown(description="Sentence index", layout=widgets.Layout(width="760px"))

truthful_generation_widget = widgets.Dropdown(
    description="Truthful sample",
    options=[("None", None)],
    value=None,
    layout=widgets.Layout(width="540px"),
)
deceptive_generation_widget = widgets.Dropdown(
    description="Deceptive sample",
    options=[("None", None)],
    value=None,
    layout=widgets.Layout(width="540px"),
)

reload_button = widgets.Button(description="Reload example")

inventory_output = widgets.Output()
summary_output = widgets.Output()
plot_output = widgets.Output()
stats_output = widgets.Output()
prefix_output = widgets.Output()
sample_output = widgets.Output()
generation_output = widgets.Output()


def get_current_probe():
    history_norm = browser_state.get("history_norm")
    sentence_idx = sentence_widget.value

    if history_norm is None or sentence_idx is None:
        return None

    return next(
        (
            probe_item for probe_item in history_norm
            if probe_item.get("sentence_idx") == sentence_idx
            or probe_item.get("sentence_end_idx") == sentence_idx + 1
        ),
        None,
    )


def update_model_options(*_):
    models_local = list_models_for_environment(
        env_widget.value,
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        token=HF_TOKEN or None,
    )
    model_widget.options = models_local
    if models_local and model_widget.value not in models_local:
        model_widget.value = models_local[0]


def update_example_options(*_):
    repo_paths_local = list_localization_repo_paths(
        env_widget.value,
        model_widget.value,
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        token=HF_TOKEN or None,
        limit=int(scan_limit_widget.value),
        filename_filter="",
    )

    example_widget.options = [(path.split("/")[-1], path) for path in repo_paths_local]

    with inventory_output:
        clear_output(wait=True)
        print(f"Environment: {env_widget.value}")
        print(f"Model: {model_widget.value}")
        print(f"Listed examples: {len(repo_paths_local):,}")
        print(f"Scan cap: {int(scan_limit_widget.value):,}")

    if repo_paths_local and example_widget.value not in repo_paths_local:
        example_widget.value = repo_paths_local[0]


def update_sample_options(*_):
    probe = get_current_probe()
    sentence_idx = sentence_widget.value
    repo_path = example_widget.value

    if probe is None or sentence_idx is None:
        truthful_generation_widget.options = [("None", None)]
        deceptive_generation_widget.options = [("None", None)]
        truthful_generation_widget.value = None
        deceptive_generation_widget.value = None
        return

    sample_selection_context = (repo_path, int(sentence_idx))
    context_changed = browser_state.get("sample_selection_context") != sample_selection_context
    browser_state["sample_selection_context"] = sample_selection_context

    truthful_options = [("None", None)]
    deceptive_options = [("None", None)]

    for sample_idx, generation in enumerate(probe.get("generations") or []):
        bucket = generation_bucket(generation)
        if bucket == "truthful":
            truthful_options.append((f"Truthful generation {sample_idx}", sample_idx))
        elif bucket == "deceptive":
            deceptive_options.append((f"Deceptive generation {sample_idx}", sample_idx))

    old_truthful = truthful_generation_widget.value
    old_deceptive = deceptive_generation_widget.value

    truthful_generation_widget.options = truthful_options
    deceptive_generation_widget.options = deceptive_options

    truthful_values = [value for _, value in truthful_options]
    deceptive_values = [value for _, value in deceptive_options]

    if context_changed or old_truthful not in truthful_values:
        truthful_generation_widget.value = None
    else:
        truthful_generation_widget.value = old_truthful

    if context_changed or old_deceptive not in deceptive_values:
        deceptive_generation_widget.value = None
    else:
        deceptive_generation_widget.value = old_deceptive


def on_truthful_change(change):
    if change.get("new") is not None and deceptive_generation_widget.value is not None:
        deceptive_generation_widget.value = None
    render_all()


def on_deceptive_change(change):
    if change.get("new") is not None and truthful_generation_widget.value is not None:
        truthful_generation_widget.value = None
    render_all()


def render_all(*_):
    record = browser_state.get("record")
    history_norm = browser_state.get("history_norm")
    df_stats = browser_state.get("df_stats")
    sentence_span_map = browser_state.get("sentence_span_map") or {}

    if record is None or history_norm is None or df_stats is None:
        return

    raw_text = record.get("raw_text") or ""
    repo_path = example_widget.value
    right_sentence_end_idx = record.get("right_sentence_end_idx")
    deceptive_sentence_idx = compute_deceptive_sentence_idx(right_sentence_end_idx, df_stats)

    with summary_output:
        clear_output(wait=True)
        display(Markdown("### Result Summary"))

        summary_lines = [
            f"- Environment: {env_widget.value}",
            f"- Model: {model_widget.value}",
            f"- Example ID: {record.get('example_id')}",
            f"- Repo path: `{repo_path}`",
            f"- Probes: {len(history_norm)}",
        ]
        if deceptive_sentence_idx is not None:
            summary_lines.append(f"- Deceptive sentence idx: {deceptive_sentence_idx}")
        if right_sentence_end_idx is not None:
            summary_lines.append(f"- Stored right sentence end idx: {right_sentence_end_idx}")

        display(Markdown("\n".join(summary_lines)))

    with plot_output:
        clear_output(wait=True)
        display(Markdown("### Deception Rate vs Sentence Index"))

        if len(df_stats) > 0:
            fig, _ = plot_sentence_localization(
                df_stats,
                deceptive_sentence_idx=deceptive_sentence_idx,
                right_sentence_end_idx=right_sentence_end_idx,
            )
            display(fig)

            import matplotlib.pyplot as plt
            plt.close(fig)
        else:
            display(Markdown("_No stats available to plot._"))

    with stats_output:
        clear_output(wait=True)
        if len(df_stats) > 0:
            display(Markdown("**Probe statistics**"))
            display(df_stats)

    sentence_idx = sentence_widget.value
    probe = get_current_probe()

    if probe is None:
        with prefix_output:
            clear_output(wait=True)
            display(Markdown("### Prefix Selector"))
            display(Markdown("_No probe found for this sentence._"))
        return

    resolved_sentence_text, resolved_start, resolved_end = resolve_sentence_span(
        sentence_idx,
        sentence_span_map,
        probe=probe,
    )

    # Prefer the sentence recovered from raw_text so this matches the dropdown label.
    # Some stored probe["sentence_text"] values can be shifted by one.
    sentence_text = (resolved_sentence_text or probe.get("sentence_text") or "").strip()

    with prefix_output:
        clear_output(wait=True)
        display(Markdown("### Prefix Selector"))
        display(
            Markdown(
                "Choose where the fixed prefix should end. The dashboard holds the text fixed up to "
                "that sentence, then shows continuations sampled from that point."
            )
        )

        print(
            f"Sentence idx: {sentence_idx} | Deception rate: {probe.get('deception_rate')} | "
            f"Valid samples: {probe.get('num_valid')}"
        )
        if sentence_text:
            display(Markdown(f"Sentence: `{sentence_text}`"))

        bucket_counts = (
            pd.Series(
                [generation_bucket(generation) for generation in (probe.get("generations") or [])],
                name="count",
            )
            .value_counts()
            .reindex(["truthful", "deceptive", "invalid"], fill_value=0)
        )
        display(bucket_counts.to_frame())

    with sample_output:
        clear_output(wait=True)
        display(Markdown("### Sample Selector"))
        display(
            Markdown(
                "Select one continuation sampled from the chosen prefix. Truthful and deceptive "
                "continuations are listed separately so you can compare how the same fixed prefix "
                "can lead to different outcomes."
            )
        )

        invalid_count = int(
            sum(
                generation_bucket(generation) == "invalid"
                for generation in (probe.get("generations") or [])
            )
        )
        if invalid_count:
            display(Markdown(f"_Invalid generations for this sentence: {invalid_count}_"))

    with generation_output:
        clear_output(wait=True)
        display(Markdown("### Generation Viewer"))
        display(
            Markdown(
                "This view shows the fixed prefix together with the selected continuation. Blue text "
                "comes before the final sentence in the fixed prefix, black text is the final sentence "
                "in the fixed prefix, and green text is the sampled continuation."
            )
        )

        selected_sample_idx = None
        if truthful_generation_widget.value is not None:
            selected_sample_idx = truthful_generation_widget.value
        elif deceptive_generation_widget.value is not None:
            selected_sample_idx = deceptive_generation_widget.value

        if selected_sample_idx is None:
            display(Markdown("_Select a truthful or deceptive sample above to view the generation._"))
            return

        generations = probe.get("generations") or []
        if selected_sample_idx >= len(generations):
            display(Markdown("_Selected sample is not available for the current sentence._"))
            return

        generation = generations[selected_sample_idx]
        gen_text = strip_code_fences(generation.get("gen_text") or "")

        # Always show the full fixed prefix ending at the selected sentence.
        if resolved_end is not None:
            prefix_text = raw_text[:resolved_end]
        else:
            prefix_text = sentence_text or ""

        prefix_mode_key = "full"

        display_selected_idx, _ = build_generation_sentence_labels(
            prefix_text,
            prefix_mode_key,
            int(sentence_idx),
        )

        # Hide sentence-index labels like ^S_12 in the rendered text.
        generation_sentence_labels = {}

        is_truthful = generation_bucket(generation) == "truthful"
        display(
            Markdown(
                f"Truthful: `{is_truthful}` | Parse error: `{generation.get('parse_error')}` | "
                f"Selected prefix ends at sentence {int(sentence_idx) + 1}"
            )
        )

        display(
            HTML(
                render_prefix_generation_with_sentence_indices_html(
                    prefix_text,
                    gen_text,
                    display_selected_idx,
                    sentence_labels=generation_sentence_labels,
                )
            )
        )


def load_current_record(*_):
    repo_path = example_widget.value
    if not repo_path:
        return

    record = load_repo_record(
        repo_path,
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        token=HF_TOKEN or None,
        cache_dir=HF_CACHE_DIR,
    )

    history_norm_local = normalize_history(record.get("history") or [])
    df_plot_local = flatten_history(history_norm_local, record.get("raw_text") or "")
    df_stats_local = build_stats(history_norm_local)
    sentence_spans_local, sentence_span_map_local = build_sentence_spans(record.get("raw_text") or "")

    browser_state["record"] = record
    browser_state["history_norm"] = history_norm_local
    browser_state["df_plot"] = df_plot_local
    browser_state["df_stats"] = df_stats_local
    browser_state["sentence_spans"] = sentence_spans_local
    browser_state["sentence_span_map"] = sentence_span_map_local
    browser_state["sample_selection_context"] = None

    sentence_options = []
    if len(df_stats_local):
        for sentence_idx in sorted(int(value) for value in df_stats_local["sentence_idx"].dropna().unique()):
            sentence_options.append(
                (
                    sentence_selector_label(sentence_idx, sentence_span_map_local),
                    sentence_idx,
                )
            )

    sentence_widget.options = sentence_options
    if sentence_options and sentence_widget.value not in [value for _, value in sentence_options]:
        sentence_widget.value = sentence_options[0][1]

    update_sample_options()
    render_all()


env_widget.observe(update_model_options, names="value")
model_widget.observe(update_example_options, names="value")
scan_limit_widget.observe(update_example_options, names="value")
example_widget.observe(load_current_record, names="value")
sentence_widget.observe(lambda change: (update_sample_options(), render_all()), names="value")
truthful_generation_widget.observe(on_truthful_change, names="value")
deceptive_generation_widget.observe(on_deceptive_change, names="value")
reload_button.on_click(lambda _: load_current_record())

update_model_options()
update_example_options()
load_current_record()

controls_top = widgets.HBox([env_widget, model_widget, scan_limit_widget])
controls_mid = widgets.HBox([reload_button])
sample_selectors = widgets.HBox([truthful_generation_widget, deceptive_generation_widget])

browser_box = widgets.VBox(
    [
        controls_top,
        controls_mid,
        inventory_output,
        example_widget,
        summary_output,
        plot_output,
        stats_output,
        prefix_output,
        sentence_widget,
        sample_output,
        sample_selectors,
        generation_output,
    ]
)

display(browser_box)

## Streamlit App

For a fuller dashboard with a cleaner interface and additional details—such as the full prompt, evaluation context, probe statistics, and expandable raw fields—you can run the Streamlit version:

```bash
streamlit run DatasetAccess/app.py